# 04 — What LayerNorm normalizes, and where it belongs

LayerNorm acts over the D features of each position, not over time or batch:
$\mathrm{LN}(x)=\gamma\odot(x-\mu)/\sqrt{\sigma^2+\epsilon}+\beta$.
Here variance divides by D (population variance), and epsilon protects small denominators.

Pre-norm writes $x+F(\mathrm{LN}(x))$; post-norm computes $\mathrm{LN}(x+F(x))$. We will inspect the distinction rather than rank architectures from one random run.

## How to work through this notebook

Run setup once. At each checkpoint, write a prediction and try the small implementation before reading its adjacent reference solution. All reference cells run unchanged from top to bottom; exercise cells contain safe, optional starting points. Numerical checks use CPU float64 unless explicitly noted. Agent-verified reference execution is separate from your learning progress.

In [ ]:
from pathlib import Path
import sys, copy, math, inspect
from dataclasses import replace
import torch
from torch import nn
from torch.nn import functional as F
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/dongxi_llms/decoder_lab.py").exists()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the Dongxi_LLMs repository")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from dongxi_llms.decoder_lab import (
    DecoderConfig, TinyDecoder, DecoderBlock, MultiHeadAttention, MLP, RMSNorm,
    layer_norm, rms_norm, rope, attend, parameter_count, analytical_parameters,
    cost_estimate, teaching_batch, next_token_loss, fit_one_batch)
torch.set_num_threads(1)
torch.manual_seed(505)
DTYPE = torch.float64
def close(actual, expected, atol=1e-10, rtol=1e-8):
    torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol)
print("CPU reference environment:", torch.__version__)


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
from dongxi_llms import decoder_visuals as viz
def show_visual(figure):
    display(figure)
    plt.close(figure)


## Architecture map — your location in the model

The highlighted stage is this lesson’s focus. B = batch, T = positions, D = model width, V = vocabulary size. This is a structural map, not measured activations or runtime. The baseline route adds learned position embeddings before the blocks.

![Architecture map — your location in the model. The highlighted stage is this lesson’s focus. B = batch, T = positions, D = model width, V = vocabulary size. This is a structural map, not measured activations or runtime. The baseline route adds learned position embeddings before the blocks.](../figures/chapter-05/day-05-04_layernorm_and_placement-architecture-map.png)

*Saved architecture schematic. The following cell regenerates it; it does not execute or train a model.*

In [ ]:
from dongxi_llms import decoder_architecture as architecture
show_visual(architecture.model_map(focus='norm', modern=False))

## Locate normalization without changing the skip path

This is pre-norm: LayerNorm processes each branch input. The residual path carries the unnormalized stream. The notebook separately contrasts post-norm behavior.

![Locate normalization without changing the skip path. This is pre-norm: LayerNorm processes each branch input. The residual path carries the unnormalized stream. The notebook separately contrasts post-norm behavior.](../figures/chapter-05/day-05-04_layernorm_and_placement-architecture-detail.png)

*Saved architecture schematic. The following cell regenerates it; it does not execute or train a model.*

In [ ]:
show_visual(architecture.block_detail(focus='norm'))

## 1. Implement and differentiate featurewise normalization

Match nn.LayerNorm using explicit mean, variance, scale, and bias. Compare gradients under a nonuniform upstream probe.

**Your prediction:** _Write it here before running the reference._

In [ ]:
x = torch.randn(2, 6, 16, dtype=DTYPE, requires_grad=True)
weight = torch.randn(16, dtype=DTYPE, requires_grad=True)
bias = torch.randn(16, dtype=DTYPE, requires_grad=True)
# Your implementation: normalized = ...

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
mean = x.mean(-1, keepdim=True)
variance = (x-mean).square().mean(-1, keepdim=True)
normalized = (x-mean)*torch.rsqrt(variance+1e-5)*weight+bias
reference = F.layer_norm(x, (16,), weight, bias, 1e-5)
close(normalized, reference)
probe = torch.randn_like(x)
ours = torch.autograd.grad((normalized*probe).sum(), (x, weight, bias))
theirs = torch.autograd.grad((reference*probe).sum(), (x, weight, bias))
for a, b in zip(ours, theirs): close(a, b)
print("Forward and input/scale/bias gradient checks passed.")
print("Before affine scale/bias, feature means:", ((x-mean)*torch.rsqrt(variance+1e-5)).mean(-1).detach())

### Why this works

Learned gamma and beta mean the final output need not have mean zero or unit variance. Epsilon also makes the normalized variance slightly below one. A uniform sum probe can hide important derivative behavior, so we used a random probe.

### Visual explanation — Separate centering from scaling

This is one position’s feature vector, before learned gamma/beta. Subtracting the mean shifts it; dividing by the RMS of the centered values rescales it. Feature indices are not semantic axes.

The figure uses this lesson’s tensors. Rerun it after changing the preceding experiment.

![Separate centering from scaling. This is one position’s feature vector, before learned gamma/beta. Subtracting the mean shifts it; dividing by the RMS of the centered values rescales it. Feature indices are not semantic axes.](../figures/chapter-05/day-05-04_layernorm_and_placement-visual-normalization.png)

*Saved reference preview. The code below regenerates this figure from the current lesson state; it does not overwrite the preview.*

In [ ]:
show_visual(viz.features({'Original': x[0,0], 'Centered': (x-mean)[0,0], 'Centered + scaled': ((x-mean)*torch.rsqrt(variance+1e-5))[0,0]}, 'LayerNorm acts across features of one position'))

## 2. Normalize the wrong axis

If normalization averages over time, can a future token change an earlier normalized state even before attention?

**Your prediction:** _Write it here before running the reference._

In [ ]:
changed = x.detach().clone(); changed[:, -1] += 10
# Your implementation: compare correct and time-axis normalization.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
def wrong_time_norm(t):
    centered = t - t.mean(1, keepdim=True)
    return centered * torch.rsqrt(centered.square().mean(1, keepdim=True)+1e-5)
w, b = torch.ones(16, dtype=DTYPE), torch.zeros(16, dtype=DTYPE)
close(layer_norm(x, w, b)[:, :-1], layer_norm(changed, w, b)[:, :-1])
assert not torch.allclose(wrong_time_norm(x)[:, 0], wrong_time_norm(changed)[:, 0])
constant = torch.full((1, 2, 16), 7., dtype=DTYPE)
close(layer_norm(constant, w, b), torch.zeros_like(constant))
print("Wrong-axis earlier-state change:",
      float((wrong_time_norm(x)[:, 0]-wrong_time_norm(changed)[:, 0]).norm().detach()))

### Why this works

Causality is a property of the whole computation, not only the attention mask. Time-axis normalization can leak future information. Epsilon keeps a constant feature vector finite; centering makes its normalized component zero.

### Visual explanation — See future leakage from the wrong normalization axis

The future input changed only at the last position. Correct featurewise normalization leaves earlier rows unchanged; time-axis normalization spreads the change backward. Both plots show differences, on one scale.

The figure uses this lesson’s tensors. Rerun it after changing the preceding experiment.

![See future leakage from the wrong normalization axis. The future input changed only at the last position. Correct featurewise normalization leaves earlier rows unchanged; time-axis normalization spreads the change backward. Both plots show differences, on one scale.](../figures/chapter-05/day-05-04_layernorm_and_placement-visual-wrong-axis.png)

*Saved reference preview. The code below regenerates this figure from the current lesson state; it does not overwrite the preview.*

In [ ]:
show_visual(viz.matrices([(layer_norm(changed,w,b)-layer_norm(x,w,b))[0], (wrong_time_norm(changed)-wrong_time_norm(x))[0]], ['Correct feature axis: change', 'Wrong time axis: change'], 'A correct attention mask cannot repair earlier normalization leakage'))

## 3. Compare pre-norm and post-norm with zero branches

Keep normalization intact and zero the learned attention/MLP branches. Which block becomes an identity?

**Your prediction:** _Write it here before running the reference._

In [ ]:
cfg = DecoderConfig()
pre = DecoderBlock(cfg).double()
post = DecoderBlock(cfg, post_norm=True).double()
# Predict before setting the branches to zero.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
for block in (pre, post):
    with torch.no_grad():
        for module in (block.attn, block.mlp):
            for parameter in module.parameters(): parameter.zero_()
pre_output, post_output = pre(x)[0], post(x)[0]
close(pre_output, x)
assert not torch.allclose(post_output, x)
print("Pre-norm identity error:", float((pre_output-x).abs().max().detach()))
print("Post-norm change:", float((post_output-x).norm().detach()))

### Why this works

The pre-norm skip path bypasses normalization. The post-norm output is normalized even when the branch contributes zero. This is an exact structural difference, not a claim that one always optimizes better.

## 4. Inspect a small stack without overgeneralizing

Copy the same weights into pre-norm and post-norm stacks. Measure activations and incoming gradients under the same probe.

**Your prediction:** _Write it here before running the reference._

In [ ]:
# Predict whether identical parameters imply identical outputs when placement changes.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
pre_stack = [DecoderBlock(cfg).double() for _ in range(4)]
post_stack = [copy.deepcopy(b) for b in pre_stack]
for b in post_stack: b.post_norm = True
probe = torch.randn_like(x)
def trace_stack(blocks):
    state = x
    norms = []
    for block in blocks:
        state = block(state)[0]
        norms.append(float(state.detach().norm()))
    gradient = torch.autograd.grad((state*probe).sum(), x)[0]
    return {"activation_norms": norms, "input_gradient_norm": float(gradient.norm())}
print("Pre-norm:", trace_stack(pre_stack))
print("Post-norm:", trace_stack(post_stack))

### Why this works

Changing placement changes the function and gradient paths even with identical weights. These are observations for one seed, depth, and loss probe—not evidence of a general training-stability ranking.

## Takeaway and evidence boundary

Next: the MLP changes features at one position; it does not directly retrieve other tokens.

Companion map: [Chapter 5 pathway](../day-05/README.md). Reusable source: [decoder_lab.py](../../src/dongxi_llms/decoder_lab.py). Record your explanation and remaining questions here; the notebook's existence does not mark the lesson complete.